# ShopAgent-AI | CP3: Agent, Memory, and Router

**Objective:** Connect the tools developed in CP2 to an LLM to create a chat interface that understands requests, invokes the correct tool, and retains conversation context. Instead of testing models in a fixed sequence, a **Router selects the model based on the type of question**.

**Models:**

| Name | Default Model | Use Case |
|---|---|---|
| `groq-fast` | `openai/gpt-oss-20b` | Simple queries (search, details, inventory) |
| `groq-smart` | `openai/gpt-oss-120b` | Complex queries (comparison, filtering, referencing previous results) |
| `ollama` | `qwen2.5:3b` | **Final fallback:** When all cloud options fail |

**How ​​does the router decide?** Using simple keyword-based rules without invoking an LLM (fast, free, and transparent). Complex queries go to the large model, simple ones to the fast model, and the local model is always the last resort.

**Memory:** All models share the same memory; if the model changes mid-conversation, the next one picks up right where the previous one left off, maintaining the same context.


In [1]:
%pip install -q python-dotenv langchain-core langgraph langchain-ollama langchain-groq 

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


## 1. Configuration
Keys are read from the `.env` file (rather than being hardcoded). Since model names often vary between companies, I made them configurable via the `.env` file so you don't have to touch the code.

In [1]:
import sys
from pathlib import Path

# Find the project root (the folder that contains "shopagent/") from wherever this notebook runs
ROOT = Path.cwd().resolve()
while not (ROOT / "shopagent").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import os
import time
import uuid

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))

GROQ_FAST_MODEL = os.getenv("GROQ_FAST_MODEL", "openai/gpt-oss-20b")  
GROQ_SMART_MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-120b")     
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "qwen2.5:3b")                 

print("GROQ_API_KEY set:", bool(os.getenv("GROQ_API_KEY")))
print("Models:", GROQ_FAST_MODEL, "|", GROQ_SMART_MODEL, "|", OLLAMA_MODEL)

GROQ_API_KEY set: True
Models: openai/gpt-oss-20b | openai/gpt-oss-120b | qwen2.5:3b


## 2.Tools
We wrap the functions in `tools.py` with the `@tool` decorator so the LLM can invoke them. The LLM reads the docstring to decide when to use the tool.

**Two important additions here:**
- **Name shortening:** Amazon product names are very long; this clutters the small local model's context and degrades its performance.
- **English search:** Product names in the data are in English, so if a customer types "kabel" (cable), the agent must translate it to "cable" before searching.

In [2]:
from langchain_core.tools import tool

from shopagent import tools as store  # our CP2 module (tools.py)


def shorten(text, limit=80):
    """Cut long text so tool results stay small for the LLM."""
    text = str(text)
    return text if len(text) <= limit else text[: limit - 3] + "..."


def short_names(rows):
    return [{**row, "name": shorten(row["name"])} for row in rows]


@tool
def search_products(
    query: str,
    max_price: float | None = None,
    min_rating: float | None = None,
    category: str | None = None,
) -> dict:
    """Search store products by keyword. Product names are in English, so translate
    the customer's request to ONE or TWO short English keywords first (for example
    'cable', 'charger', 'headphones'). Only use max_price, min_rating or category
    if the customer explicitly asked for them. Prices are in INR, rating is 0-5.
    Returns up to 5 products with id, name, price, stock, rating."""
    if not query.strip():
        return {"count": 0, "products": [], "message": "Empty search. Provide a keyword."}

    products = short_names(store.search_products(query, max_price, min_rating, category))
    if not products:
        return {
            "count": 0,
            "products": [],
            "message": "No products found. Try a shorter English keyword or remove the filters.",
        }
    return {"count": len(products), "products": products}


@tool
def get_product_details(product_id: int) -> dict:
    """Get full details (brand, price, stock, rating, description) of one product by its id."""
    details = store.get_product_details(product_id)
    if "error" not in details:
        details["name"] = shorten(details["name"])
        details["description"] = shorten(details.get("description") or "", 300)
    return details


@tool
def compare_products(product_ids: list[int]) -> dict:
    """Compare 2 to 4 products by id on price, rating and stock.
    Also tells which one is the cheapest and which one is the best rated."""
    result = store.compare_products(product_ids)
    if "products" in result:
        result["products"] = short_names(result["products"])
    return result


@tool
def check_stock(product_id: int, quantity: int = 1) -> dict:
    """Check if the requested quantity of a product (by id) is available in stock."""
    result = store.check_stock(product_id, quantity)
    if "name" in result:
        result["name"] = shorten(result["name"])
    return result


TOOLS = [search_products, get_product_details, compare_products, check_stock]
print("Tools ready:", [t.name for t in TOOLS])

Tools ready: ['search_products', 'get_product_details', 'compare_products', 'check_stock']


## 3.models


In [3]:
def build_llms():
    """Create every model that can be created. A missing key just skips that model."""
    llms = {}

    # Groq: two sizes behind one key (needs GROQ_API_KEY)
    try:
        from langchain_groq import ChatGroq

        llms["groq-fast"] = ChatGroq(model=GROQ_FAST_MODEL, temperature=0)
        llms["groq-smart"] = ChatGroq(model=GROQ_SMART_MODEL, temperature=0)
    except Exception as exc:
        print(f"[warn] groq not available: {exc}")

    # Local model: last-resort fallback (Ollama app must be running)
    try:
        from langchain_ollama import ChatOllama

        llms["ollama"] = ChatOllama(model=OLLAMA_MODEL, temperature=0)
    except Exception as exc:
        print(f"[warn] ollama not available: {exc}")

    return list(llms.items())


LLMS = build_llms()
print("Models created:", [name for name, _ in LLMS])

Models created: ['groq-fast', 'groq-smart', 'ollama']


## 4. Health check


In [5]:
def check_models(llms):
    for name, llm in llms:
        start = time.time()
        try:
            llm.invoke("Reply with the single word OK")
            print(f"[OK]   {name:<8} {time.time() - start:5.1f}s")
        except Exception as exc:
            print(f"[FAIL] {name:<8} {type(exc).__name__}: {str(exc)[:150]}")


check_models(LLMS)

[OK]   groq-fast   3.8s
[OK]   groq-smart   9.4s
[OK]   ollama     1.9s


## 5. Agents and Memory
- **System prompt:** Defines the agent's behavioral rules (e.g., do not hallucinate, respond in the customer's language, reference previously presented products).
- **MemorySaver:** Stores the conversation for each `thread_id` and is shared across all three models.

> If a deprecation warning appears regarding `create_react_agent`, it is nothing to worry about; LangChain libraries update rapidly, and the code will still function correctly.

In [6]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent

SYSTEM_PROMPT = """You are ShopAgent, a helpful assistant for an online store.

Rules:
- Always use the tools to look up products. Never invent products, prices or stock.
- Product names in the database are in English. Translate the customer's request
  to a short English keyword before searching, but reply in the customer's language
  (Arabic or English).
- When you list products, show the id, a short name, the price and the rating.
- When the customer says "the first", "the second" or "the last one", refer to the
  products you listed in your previous message and use their ids.
- Prices are in Indian rupees (INR) unless told otherwise.
- You cannot place orders yet. If asked to buy, say ordering is coming soon.
- If a tool returns nothing or an error, say so honestly and suggest another search.
- If the question is not about the store, politely say you can only help with shopping.
"""

MEMORY = MemorySaver()  # one memory shared by all agents

AGENTS = [
    (name, create_react_agent(llm, TOOLS, checkpointer=MEMORY, prompt=SYSTEM_PROMPT))
    for name, llm in LLMS
]
print("Agents ready:", [name for name, _ in AGENTS])

Agents ready: ['groq-fast', 'groq-smart', 'ollama']


C:\Users\GPU-TECH\AppData\Local\Temp\ipykernel_20776\484129327.py:23: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  (name, create_react_agent(llm, TOOLS, checkpointer=MEMORY, prompt=SYSTEM_PROMPT))


## 6. The Router and the `chat()` Function
The **Router** classifies the query as either `simple` or `complex` based on keyword rules (without invoking an LLM) and then prioritizes the models according to the type:

| Query Type | Priority Order |
|---|---|
| `complex` (comparison, recommendation, "first"/"second", long query, multiple numbers) | groq-smart →  → groq-fast → **ollama** |
| `simple` (search, details, or simple inventory lookup) | groq-fast →  → groq-smart → **ollama** |

The local model (`ollama`) is always last, serving as the fallback. If a model fails with an error, `chat()` proceeds to the next one. The `skip` parameter is used to test the fallback mechanism.

In [7]:
import re


def as_text(content):
    """Some models return a list of content blocks instead of a plain string."""
    if isinstance(content, list):
        return "".join(
            part.get("text", "") if isinstance(part, dict) else str(part) for part in content
        )
    return str(content)


# ---------------- Router ----------------
def norm(text):
    """Lowercase and unify Arabic letter variants so keyword matching is forgiving."""
    text = text.lower()
    for a, b in (("أ", "ا"), ("إ", "ا"), ("آ", "ا"), ("ى", "ي"), ("ة", "ه")):
        text = text.replace(a, b)
    return text


# Words that signal a task needing more reasoning
COMPLEX_WORDS = [norm(w) for w in (
    "قارن", "مقارنه", "الفرق", "الافضل", "افضل", "احسن", "رشح", "رشحلي", "انصحني",
    "compare", "difference", "versus", "vs", "best", "recommend", "better", "which one",
)]
# Words that point back to earlier results (the model must use the conversation memory)
REFERENCE_WORDS = [norm(w) for w in (
    "الاول", "التاني", "الثاني", "التالت", "الثالث", "الاخير", "ده", "دي",
    "first", "second", "third", "last one", "that one", "previous",
)]


def has_any(text, words):
    tokens = set(re.findall(r"\w+", text))
    return any((w in text) if (" " in w or len(w) >= 4) else (w in tokens) for w in words)


def classify(message):
    """Return 'complex' or 'simple' using cheap keyword rules (no LLM call)."""
    text = norm(message)
    if has_any(text, COMPLEX_WORDS) or has_any(text, REFERENCE_WORDS):
        return "complex"
    if len(text) > 120 or len(re.findall(r"\d+", text)) >= 2:
        return "complex"
    return "simple"


ROUTES = {
    "complex": ["groq-smart", "groq-fast", "ollama"],
    "simple": ["groq-fast", "groq-smart", "ollama"],
}


def route(message):
    """Return (task_kind, ordered list of available models). Ollama is always last."""
    kind = classify(message)
    available = {name for name, _ in AGENTS}
    return kind, [name for name in ROUTES[kind] if name in available]


# ---------------- Chat with fallback ----------------
def chat(message, thread_id="demo", skip=()):
    config = {"configurable": {"thread_id": thread_id}}
    kind, order = route(message)
    agents = dict(AGENTS)
    errors = []

    for name in order:
        if name in skip:
            continue
        start = time.time()
        try:
            result = agents[name].invoke({"messages": [("user", message)]}, config)
        except Exception as exc:  # connection, rate limit, wrong key, ...
            errors.append(f"{name}: {type(exc).__name__}: {str(exc)[:150]}")
            print(f"[warn] {name} failed -> trying the next model")
            continue

        messages = result["messages"]
        last_human = max(i for i, m in enumerate(messages) if m.type == "human")
        tools_used = [
            call["name"]
            for m in messages[last_human + 1 :]
            for call in (getattr(m, "tool_calls", None) or [])
        ]
        return {
            "reply": as_text(messages[-1].content),
            "model": name,
            "task": kind,
            "seconds": round(time.time() - start, 1),
            "tools": tools_used,
        }

    raise RuntimeError("All models failed:\n" + "\n".join(errors))


def show(result):
    print(f"[{result['task']} -> {result['model']} | {result['seconds']}s | tools: {result['tools']}]")
    print(result["reply"])

### 6.1 Router Test (Without Any API Calls)
We observe how the Router directs each question to the appropriate model. This step is free and fast, as it does not call any model.

In [8]:
samples = [
    "عاوز كابل شحن",
    "Show me USB cables under 500",
    "عاوز كابل شحن تحت 500 وتقييمه فوق 4",
    "قارن بين الأول والتاني",
    "التاني ده متوفر منه 3؟",
    "Which one is better, 1512 or 1660?",
    "رشحلي سماعات",
    "مرحبا",
]
for text in samples:
    kind, order = route(text)
    print(f"{kind:<8} -> {order[0]:<10} (then {order[1:]}) | {text}")

simple   -> groq-fast  (then ['groq-smart', 'ollama']) | عاوز كابل شحن
simple   -> groq-fast  (then ['groq-smart', 'ollama']) | Show me USB cables under 500
complex  -> groq-smart (then ['groq-fast', 'ollama']) | عاوز كابل شحن تحت 500 وتقييمه فوق 4
complex  -> groq-smart (then ['groq-fast', 'ollama']) | قارن بين الأول والتاني
complex  -> groq-smart (then ['groq-fast', 'ollama']) | التاني ده متوفر منه 3؟
complex  -> groq-smart (then ['groq-fast', 'ollama']) | Which one is better, 1512 or 1660?
complex  -> groq-smart (then ['groq-fast', 'ollama']) | رشحلي سماعات
simple   -> groq-fast  (then ['groq-smart', 'ollama']) | مرحبا


## 7. Test Conversation
The same `thread_id` is used across all messages, so the agent needs to remember the "first" and the "second" ones.

In [9]:
thread = str(uuid.uuid4())

for question in [
    "عاوز كابل شحن تحت 500",
    "قارن بين الأول والتاني",
    "التاني ده متوفر منه 3؟",
]:
    print(f"\nYou: {question}")
    show(chat(question, thread))


You: عاوز كابل شحن تحت 500
[simple -> groq-fast | 4.6s | tools: ['search_products']]
إليك بعض كابلات الشحن التي تكلف أقل من 500 روبيه:

| ID | اسم مختصر | السعر (₹) | التقييم |
|----|-----------|-----------|---------|
| 1512 | Syncwire USB‑C Cable | 399 | 5.0 |
| 1660 | REDTECH USB‑C to Lightning | 249 | 5.0 |
| 1368 | AmazonBasics USB‑2.0 Extension | 199 | 4.5 |
| 1378 | AmazonBasics USB‑2.0 A‑Male to B‑Male | 209 | 4.5 |
| 1408 | Oraimo 65W USB‑C to USB‑C | 349 | 4.5 |

إذا أردت تفاصيل أكثر عن أي منتج أو مقارنة بينهما، أخبرني بالـ ID.

You: قارن بين الأول والتاني
[complex -> groq-smart | 6.1s | tools: ['compare_products']]
**مقارنة بين الكابلين**

| الخاصية | الكابل الأول (ID 1512) | الكابل الثاني (ID 1660) |
|--------|------------------------|--------------------------|
| **السعر** | 399 ₹ | **249 ₹** (الأرخص) |
| **التقييم** | 5.0 (الأعلى تقييمًا) | 5.0 (متساوي) |
| **المخزون** | 41 وحدة | 5 وحدات |

- **الأرخص**: الكابل الثاني (REDTECH) بسعر 249 ₹.  
- **الأعلى تقييمًا**: كلاهما 

## 8. Mini-evaluation
Instead of relying on a subjective "eyeball test," we run a few sample questions to see whether the agent invokes the expected tool. This provides a metric you can share with the team, which can be re-run whenever the prompt or model is updated.



In [10]:
EVAL_CASES = [
    ("I want a charging cable under 500 rupees", "search_products"),
    ("عاوز سماعات بلوتوث", "search_products"),
    ("Tell me more about product 1512", "get_product_details"),
    ("Is product 1512 available in quantity 3?", "check_stock"),
    ("Compare products 1512 and 1660", "compare_products"),
    ("What is the weather today?", None),  
]

passed = 0
for question, expected in EVAL_CASES:
    result = chat(question, thread_id=str(uuid.uuid4()))
    ok = (expected in result["tools"]) if expected else (not result["tools"])
    passed += ok
    print(f"{'PASS' if ok else 'FAIL'}  [{result['model']}] expected={expected} used={result['tools']} | {question}")

print(f"\nTool-selection accuracy: {passed}/{len(EVAL_CASES)}")

PASS  [groq-fast] expected=search_products used=['search_products'] | I want a charging cable under 500 rupees
PASS  [groq-fast] expected=search_products used=['search_products'] | عاوز سماعات بلوتوث
PASS  [groq-fast] expected=get_product_details used=['get_product_details'] | Tell me more about product 1512
PASS  [groq-smart] expected=check_stock used=['check_stock'] | Is product 1512 available in quantity 3?
PASS  [groq-smart] expected=compare_products used=['compare_products'] | Compare products 1512 and 1660
PASS  [groq-fast] expected=None used=[] | What is the weather today?

Tool-selection accuracy: 6/6


## 9. Fallback Test
Instead of manually breaking the API keys, we configure `chat()` to skip specific models:
1. Skip all cloud models → The **local** model must respond (this is the final fallback).
2. Skip `groq-smart` for a complex query → The next model in the sequence must respond.

> The first test might take some time (due to the initial loading of the local model from the disk).

In [11]:
print("1) All cloud models skipped -> the local model must answer:")
show(chat("Show me HDMI cables", "fallback-1", skip={"groq-fast", "groq-smart"}))

print("\n2) Preferred model skipped -> the next one must answer:")
show(chat("Compare products 1512 and 1660", "fallback-2", skip={"groq-smart"}))

1) All cloud models skipped -> the local model must answer:
[simple -> ollama | 99.5s | tools: ['search_products']]
It seems there was an issue with the search parameters. Let's try searching for HDMI cables directly. Could you please specify a price range or any specific features you're looking for? If not, I will show you some HDMI cables within a reasonable price range.

2) Preferred model skipped -> the next one must answer:
[complex -> groq-fast | 1.8s | tools: ['compare_products']]
Here’s a quick comparison of the two products:

| Product ID | Name | Price (INR) | Rating | Stock |
|------------|------|-------------|--------|-------|
| 1512 | Syncwire LTG to USB Cable for Fast Charging | ₹399 | ⭐5.0 | 41 |
| 1660 | REDTECH USB‑C to Lightning Cable (Apple MFi Certified) | ₹249 | ⭐5.0 | 5 |

**Summary**

- **Cheapest**: Product 1660 (₹249)  
- **Best‑rated**: Both are rated ⭐5.0, so either is top‑rated.  
- **Stock**: Product 1512 has more availability (41 units) compared to Product

## 10. Interactive Chat 
Type `exit` to quit and `new` for a new conversation.

In [12]:
thread = str(uuid.uuid4())
while True:
    text = input("\nYou: ").strip()
    if text.lower() == "exit":
        break
    if text.lower() == "new":
        thread = str(uuid.uuid4())
        print("(new conversation)")
        continue
    if text:
        try:
            show(chat(text, thread))
        except RuntimeError as exc:
            print(exc)

[simple -> groq-fast | 23.3s | tools: ['search_products']]
Here are some USB cables we have in stock:

| ID | Short name | Price (₹) | Rating |
|----|------------|-----------|--------|
| 1512 | Syncwire LTG USB Cable | 399 | 5.0 |
| 1660 | REDTECH USB‑C to Lightning Cable | 249 | 5.0 |
| 1358 | Duracell USB Lightning Cable | 970 | 4.5 |
| 1368 | AmazonBasics USB 2.0 A‑Male to A‑Female | 199 | 4.5 |
| 1378 | AmazonBasics USB 2.0 A‑Male to B‑Male | 209 | 4.5 |

Let me know if you’d like details on any of these, or if you’d like to compare a few of them.
[simple -> groq-fast | 6.3s | tools: ['search_products', 'search_products']]
عذرًا، لا يوجد شاحن Type‑C في قاعدة بياناتنا حالياً. يمكننا البحث عن “USB‑C charger” أو “Type‑C power adapter” إذا كان ذلك يناسبك. هل ترغب في تجربة بحث آخر؟
[simple -> groq-fast | 4.3s | tools: ['search_products', 'search_products']]
إليك بعض الشواحن المتوفرة لدينا:

| ID | اسم مختصر | السعر (₹) | التقييم |
|----|-----------|-----------|---------|
| 2138 | Lenovo